# K_00 – Business Strategy

**Grid-Arbitrage** · Batteriespeicher-Arbitrage im Schweizer Strommarkt (Kür)

**Gruppe:** SC26_Gruppe_2 | **Verantwortlich:** Patrik Neunteufel | **Datum:** März 2026

---

*Strategische Gesamtsicht: Marktlage, Segmentstrategie, Standort, Skalierung, [VPP](../organisation/O_02_Glossar.ipynb#g-vpp).*


| [← K_99 – Kombinierte Simulation](K_99_Kombinierte_Simulation.ipynb) | [↑ Übersicht ↑](../organisation/O_01_Project_Overview.ipynb) | [K_01 – Räumliche Analyse →](K_01_Raeumliche_Analyse.ipynb) |
|:---|:---:|---:|

## Inhaltsverzeichnis<a id='toc_K_00'></a>

1. [1. Marktlage: Der Schweizer Strommarkt als Arbitrage-Umfeld](#1-marktlage-der-schweizer-strommarkt-als-arbitrage-umfeld_K_00)
2. [2. Wertquellen: Erlösstacking](#2-wertquellen-erloesstacking_K_00)
3. [3. Segmentstrategie: Wer profitiert wirklich?](#3-segmentstrategie-wer-profitiert-wirklich_K_00)
4. [4. Timing-Strategie: Wann ist der Batterieeinsatz am wertvollsten?](#4-timing-strategie-wann-ist-der-batterieeinsatz-am-wertvollsten_K_00)
5. [5. Standortstrategie: Wo ist der Battery Value am höchsten?](#5-standortstrategie-wo-ist-der-battery-value-am-hoechsten_K_00)
6. [6. Skalierungsszenarien: Vom Einzelprojekt zum Virtual Power Plant](#6-skalierungsszenarien-vom-einzelprojekt-zum-virtual-power-plant_K_00)
7. [7. Virtual Power Plant (VPP) — Systemwirkung & Privat-Business-Case](#7-virtual-power-plant-vpp-systemwirkung-privat-business-case_K_00)
8. [8. Empirische Validierung](#8-empirische-validierung_K_00)
9. [9. Handlungsempfehlungen nach Stakeholder](#9-handlungsempfehlungen-nach-stakeholder_K_00)
10. [10. Kombinationsmatrix: Parameter-Szenarien](#10-kombinationsmatrix-parameter-szenarien_K_00)
11. [11. Strategisches Fazit](#11-strategisches-fazit_K_00)
12. [Abschluss](#abschluss_K_00)


## Initialisierung<a id='initialisierung_K_00'></a>

[↑ Inhaltsverzeichnis](#toc_K_00)

Bibliotheken laden, `../sync/config.json` lesen, Verzeichnispfade setzen.

**Imports und Versionen:**

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
# K_00 ist ein reiner Bericht — er zeigt Charts und Animationen aus anderen NBs.
# Animationen erzeugen: NB08a_Animationen.ipynb (Kür) zuerst ausführen.
import os, json, pandas as pd
from IPython.display import display, Image
from datetime import datetime

# Versionen anzeigen für Reproduzierbarkeit
print(f"📅 Zuletzt ausgeführt am: {datetime.now().strftime('%d.%m.%Y um %H:%M:%S')}")


**Setup – Konfiguration & Verzeichnisstruktur:** Lädt `../sync/config.json` (SSOT), setzt Pfade.

In [ ]:
with open('../sync/config.json') as _f:
    CFG = json.load(_f)

SZ_AKTIV     = CFG['szenarien']['gleichzeitigkeit_aktiv']
DIR_INTER    = os.path.join('../data', 'intermediate')
DIR_INTER_SZ = os.path.join(DIR_INTER, SZ_AKTIV)
CHARTS_DIR   = os.path.join('../output', 'charts', SZ_AKTIV)   # Pflicht (NB03)

def show(filename, caption='', width=1100):
    path = os.path.join(CHARTS_DIR, filename)
    if not os.path.exists(path):
        print(f'Nicht vorhanden: {path}')
        return
    display(Image(filename=path, width=width))
    if caption: print(f'\n{caption}\n')


def show_anim(filename, caption='', width=1100):
    path = os.path.join(CHARTS_DIR, filename)
    if not os.path.exists(path):
        print(f'Nicht vorhanden: {path} — K_04 zuerst ausführen.')
        return
    from IPython.display import HTML
    display(HTML(f'<img src="{path}" width="{width}">'))
    if caption: print(f'\n{caption}\n')

# Gleichzeitigkeit aus NB02-Ergebnis lesen
SZ_FILE   = os.path.join(DIR_INTER_SZ, 'netzentlastung_szenarien.csv')
_gz_mode  = 'unbekannt'
_gz_rate  = '?'
if os.path.exists(SZ_FILE):
    _df_sz = pd.read_csv(SZ_FILE)
    if 'gleichzeitigkeit' in _df_sz.columns:
        _gz_mode = _df_sz['gleichzeitigkeit'].iloc[0]
        _gz_rate = f"{_df_sz['rate_pct'].iloc[0]:.0f}%"
        print(f'Szenarien geladen — Gleichzeitigkeit: {_gz_mode} ({_gz_rate})')
    else:
        print('Szenarien geladen (älteres Format)')
else:
    print('netzentlastung_szenarien.csv fehlt → NB02 zuerst ausführen')

print(f'Setup OK | Szenario={SZ_AKTIV}')
print(f'  Pflicht-Charts : {CHARTS_DIR}')
print(f'  Animationen    : {CHARTS_DIR}')
# -- Kür-Aktiv Flags aus ../sync/config.json ----------------------------------------
KUER_AKTIV = CFG.get('kuer_aktiv', {})
FORCE_RELOAD = CFG.get('force_reload', {})  # konventionskonform gelesen

def nb_aktiv(nb_key):
    """True wenn dieses Kür-Notebook eingereicht wird."""
    return KUER_AKTIV.get(nb_key, True)

def check_aktiv(*nb_keys):
    """Zeigt UNGÜLTIG-Banner wenn mindestens ein benötigtes NB nicht aktiv.
    Gibt True zurück wenn alles aktiv, False wenn nicht.
    Aufruf am Anfang jeder abhängigen Sektion: if not check_aktiv('K_01'): pass
    """
    inaktiv = [k for k in nb_keys if not KUER_AKTIV.get(k, True)]
    if inaktiv:
        print('─'*70)
        print(f'⚠️  ABSCHNITT NICHT GÜLTIG')
        print(f'   Benötigte Notebooks nicht eingereicht: {", ".join(inaktiv)}')
        print(f'   Inhalt dieses Abschnitts basiert auf nicht verfügbaren Analysen.')
        print(f'   Aktivieren: ../sync/config.json → kuer_aktiv.{inaktiv[0]} = true')
        print('─'*70)
        return False
    return True

# Statusübersicht
aktiv_list   = [k for k,v in KUER_AKTIV.items() if v]
inaktiv_list = [k for k,v in KUER_AKTIV.items() if not v]
print(f'Kür aktiv ({len(aktiv_list)}): {", ".join(aktiv_list)}')
if inaktiv_list:
    print(f'Kür INAKTIV ({len(inaktiv_list)}): {", ".join(inaktiv_list)}')
    print('  → Betroffene Abschnitte in K_00 werden als NICHT GÜLTIG markiert')
# -- Transfer: Ergebnisse aus ../sync/transfer.json laden ----------------------------
_tf_path = '../sync/transfer.json'
if os.path.exists(_tf_path):
    TF = json.load(open(_tf_path)) if os.path.exists(_tf_path) and os.path.getsize(_tf_path) > 0 else {}
    _dt     = TF.get('datenzeitraum', {})
    _sim    = TF.get('simulation', {})
    TF_N_YEARS  = _dt.get('n_years', None)
    TF_START    = _dt.get('start_date', 'unbekannt')
    TF_END      = _dt.get('end_date', 'unbekannt')
    TF_SPREAD   = _sim.get('spread_mean_eur_mwh', None)
    TF_ECON     = _sim.get('wirtschaftlichkeit', {})
    TF_HYB      = TF.get('hybrid_simulation', {}).get('ergebnisse', {})
    TF_KUER     = CFG.get('kuer_aktiv', {})   # aus ../sync/config.json (SSOT)
    TF_DISP     = TF.get('dispatch_optimierung', {})   # T4: K_06-Ergebnisse
    TF_PROD     = TF.get('produkt', {})                # T5: K_10-Produktdaten
    print(f"../sync/transfer.json: {TF_START} – {TF_END} ({TF_N_YEARS} Jahre) | Spread: {TF_SPREAD} EUR/MWh")
else:
    TF = {}; TF_N_YEARS = None; TF_START = TF_END = 'unbekannt'
    TF_SPREAD = None; TF_ECON = {}; TF_HYB = {}; TF_KUER = {}
    TF_DISP = {}; TF_PROD = {}
    print('⚠️  ../sync/transfer.json nicht gefunden — NB01/NB02 zuerst ausführen')

---
## 1. Marktlage: Der Schweizer Strommarkt als Arbitrage-Umfeld <a id='1-marktlage-der-schweizer-strommarkt-als-arbitrage-umfeld_K_00'></a>

[↑ Inhaltsverzeichnis](#toc_K_00)

### 1.1 Preisstruktur

Der Schweizer Day-Ahead Markt zeigt eine ausgeprägte Tages- und Saisonalstruktur die Arbitrage systematisch ermöglicht. Zwei Effekte prägen das Bild:

- **Tagestief morgens und im Sommer-Mittag** durch Basislast-Überhang und Solarproduktion
- **Preissspitzen abends** durch Haushalts- und Industrielast in der Dämmerung

Der simulierbare Intra-Tag-Spread (p75−p25) beträgt im Jahresdurchschnitt 20–40 EUR/MWh — dies ist die Grundlage der Simulation und des Rentabilitäts-Triggers (> 30 EUR/MWh, vgl. K_03). Davon zu unterscheiden ist die rohe Marktpreisspanne (max−min): Chart 5a zeigt den überraschenden Befund, dass **Frühling** die höchste Marktpreisspanne aufweist (~139 EUR/MWh, max−min), Winter die niedrigste (~85 EUR/MWh) — der Duck-Curve-Effekt wirkt im Frühling am stärksten. Im Sommer-Mittag entstehen durch Solarüberproduktion Negativpreise — ideale kostenlose Ladezyklen.


In [ ]:
check_aktiv('K_04')  # Kür-Abhängigkeit prüfen



In [ ]:
show('nb04_heatmap_preis.png',
     'Marktstruktur: Ø Spot-Preis CH 2023–2024 — Stunde × Monat', width=900)


### 1.2 Saisonaler Jahresverlauf

Die Animation zeigt wie sich Preisniveau, Netzlast und Arbitrage-Spread über 52 Kalenderwochen entwickeln. Deutlich erkennbar: der Frühling hat den höchsten Arbitrage-Spread — danach fällt er im Sommer ab, der dafür Negativpreis-Stunden als Ladezyklen ohne Kosten bietet.


In [ ]:
show_anim('kuer_k04_anim_C_spread.gif',
          'Animation C: Arbitrage-Spread, Negativpreise & Dispatch-Effizienz über 52 Wochen',
          width=900)


---
## 2. Wertquellen: Erlösstacking <a id='2-wertquellen-erloesstacking_K_00'></a>

[↑ Inhaltsverzeichnis](#toc_K_00)

Eine Batterie verdient aus mehreren Quellen gleichzeitig. Die Kombination bestimmt die tatsächliche Wirtschaftlichkeit:

| Erlösquelle | Beschreibung | Wer kann teilnehmen |
|-------------|-------------|--------------------|
| **Grid-Arbitrage** | Laden günstig, Einspeisen teuer | Alle Segmente |
| **Netzentlastung** | Peak-Shaving zur Spitzenlastzeit | Gewerbe, Industrie, Utility |
| **FCR/aFRR** | Regelenergiemarkt Frequenzhaltung | Industrie, Utility (>1 MW) |
| **Eigenverbrauch** | EV, Wärmepumpe, Photovoltaik kombiniert | Privat, Gewerbe |
| **Negativpreise** | Strom gratis oder mit Gutschrift laden | Alle Segmente |

**Nur [Grid-Arbitrage](../organisation/O_02_Glossar.ipynb#g-grid-arbitrage) allein** reicht bei Privat/Gewerbe meist nicht für attraktiven [ROI](../organisation/O_02_Glossar.ipynb#g-roi). Ab Industrie-Skala wird es interessant — und Utility ist mit direktem Marktzugang klar rentabel.


In [ ]:
# ROI reine Arbitrage — Ausgangslage für Stacking-Überlegung
show('nb04_roi.png',
     'Basis-ROI aus reiner Arbitrage — vor Erlösstacking', width=850)
# Vergleich alle Modelle inkl. Eigenverbrauch und Hybrid (K_99)
show('kuer_k99_roi_vergleich.png',
     'ROI-Vergleich: Arbitrage / Eigenverbrauch / Hybrid statisch / Hybrid optimiert', width=950)
show('kuer_k99_breakeven.png',
     'Break-Even-Vergleich: Vier Modelle x Vier Segmente', width=950)
show('kuer_k99_nutzerverhalten.png',
     'Robustheit Hybrid gegen Verbrauchsabweichung -- Privat 10 kWh', width=950)



---
## 3. Segmentstrategie: Wer profitiert wirklich? <a id='3-segmentstrategie-wer-profitiert-wirklich_K_00'></a>

[↑ Inhaltsverzeichnis](#toc_K_00)

### 3.1 Amortisation

Die Amortisationskurven zeigen wie lange das eingesetzte Kapital gebunden ist. Bei reiner [Grid-Arbitrage](../organisation/O_02_Glossar.ipynb#g-grid-arbitrage) (2023/2024) erreicht kein Segment den Break-Even innerhalb der modellierten Nutzungsdauer — das zeigt NB04 eindeutig. Erst mit [Erlösstacking](../organisation/O_02_Glossar.ipynb#g-erloess-stacking) (FCR/aFRR, Eigenverbrauch, Peak-Shaving) wird Industrie/Utility rentabel; Privat/Gewerbe brauchen zusätzlich sinkende [CAPEX](../organisation/O_02_Glossar.ipynb#g-capex) und/oder einen höheren Marktspread.


In [ ]:
show('nb04_amortisation.png',
     'Kumulierter Cashflow 2023–2024 (nur Arbitrage)', width=900)


### 3.2 Segmentempfehlungen

| Segment | Strategie | Zusatzerlös nötig? | Empfehlung |
|---------|----------|-------------------|------------|
| **Privat 10 kWh** | Eigenverbrauchsopt. + Arbitrage + Peak-Shaving | Ja | EV/WP-Kombination; → [K_09](K_09_Eigenverbrauch.ipynb) |
| **Gewerbe 100 kWh** | Lastspitzenvermeidung + Arbitrage | Teilweise | Netzentgeltsenkung nutzen |
| **Industrie 1 MWh** | Arbitrage + FCR/aFRR | Nein (mit FCR) | Direkte Vermarktung |
| **Utility 10 MWh** | Vollmarktintegration, [BVI](../organisation/O_02_Glossar.ipynb#g-bvi)-optimiert | Nein | Standort nach BVI wählen |


In [ ]:
show('nb04_erloese_kwh.png', 'Erlös/kWh normiert — Effizienzvergleich', width=800)
show('nb04_capex_ertrag.png', 'CAPEX vs. 12J-Netto-Erlös', width=800)


---
## 4. Timing-Strategie: Wann ist der Batterieeinsatz am wertvollsten? <a id='4-timing-strategie-wann-ist-der-batterieeinsatz-am-wertvollsten_K_00'></a>

[↑ Inhaltsverzeichnis](#toc_K_00)

### 4.1 Tageszeit

Die Abendstunden (17–20 Uhr) sind die rentabelsten Einspeisestunden. Die Morgenstunden (02–06 Uhr) und das Solarmittagsfenster (10–14 Uhr im Sommer) sind die optimalen Ladefenster. Ein saisonal adaptierter [Dispatch](../organisation/O_02_Glossar.ipynb#g-dispatch)-Algorithmus verschiebt das Ladefenster im Sommer von Nacht auf Mittag.


In [ ]:
check_aktiv('K_04')  # Kür-Abhängigkeit prüfen



In [ ]:
show('nb04_tagesprofil_einzel.png',
     'Ø Tagesprofil: Netzlast & Preis mit Lade/Einspeisefenstern', width=900)


### 4.2 Saisonale Dispatch-Optimierung

Alle vier Jahreszeiten bieten unterschiedliche Chancen — eine Beschränkung auf "Winter vs. Sommer" würde den wichtigsten Befund verfehlen:

| Saison | Marktpreisspanne (max−min) | Optimales Ladefenster | Strategie |
|--------|---------------------------|-----------------------|-----------|
| **Frühling** | **~139 EUR/MWh** (höchste) | Nacht + früher Morgen | [Dispatch](../organisation/O_02_Glossar.ipynb#g-dispatch)-Priorität: Frühling maximiert Erlös |
| **Herbst** | ~120 EUR/MWh | Nacht | Standard-Dispatch |
| **Sommer** | ~100 EUR/MWh | **Solar-Mittag** (Negativpreise!) | Ladefenster auf 10–14h verschieben |
| **Winter** | ~85 EUR/MWh (niedrigste) | Frühe Morgenstunden | Geringster Arbitrage-Erlös — entgegen Intuition |

Der Frühlingsabend (19:00) ist der rentabelste Einspeisezeitpunkt im Jahr — der Duck-Curve-Effekt ist maximal, weil Solar bereits aktiv ist aber Heizlast noch hoch. Im Sommer verschiebt sich das Ladefenster auf den Mittag — bei Negativpreisen sogar mit Vergütung.

**Ein saisonal adaptiver Dispatch-Algorithmus kann den Jahres-[ROI](../organisation/O_02_Glossar.ipynb#g-roi) um 10–20% steigern** gegenüber einem statischen Dispatch der immer dasselbe Zeitfenster verwendet.


In [ ]:
show('nb04_saisonal_frühling.png', 'Frühling: höchster Intraday-Spread (~139 EUR/MWh) — Duck-Curve-Effekt maximal', width=900)
show('nb04_saisonal_sommer.png', 'Sommer: Solar-Mittagstief, Negativpreise als Ladezyklen', width=900)



### 4.3 Jahresverlauf nach Tageszeit

Die vier Animationen zeigen wie sich die kritischen Tageszeiten über 52 Wochen verändern. Besonders aufschlussreich: 12:00 Uhr kippt von Winter (teuer) zu Sommer (günstig/negativ) — das Ladefenster wandert mit der Sonne.


In [ ]:
# Morgenspitze
show_anim('kuer_k04_anim_A_07h.gif',
          'Animation A — 07:00 Uhr: Morgenspitze über 52 Wochen', width=900)


In [ ]:
# Solarmittag
show_anim('kuer_k04_anim_A_12h.gif',
          'Animation A — 12:00 Uhr: Solar-Mittagstief über 52 Wochen', width=900)


In [ ]:
# Abendspitze
show_anim('kuer_k04_anim_A_19h.gif',
          'Animation A — 19:00 Uhr: Abendspitze über 52 Wochen', width=900)


In [ ]:
# Kombiniert alle 4
show_anim('kuer_k04_anim_B_4panel.gif',
          'Animation B — Alle 4 Tageszeiten gleichzeitig: 00h / 07h / 12h / 19h', width=950)


---
## 5. Standortstrategie: Wo ist der Battery Value am höchsten? <a id='5-standortstrategie-wo-ist-der-battery-value-am-hoechsten_K_00'></a>

[↑ Inhaltsverzeichnis](#toc_K_00)

### 5.1 Zonenimbalance

Die Schweiz ist keine homogene Stromzone. Fünf Netzregionen haben fundamental unterschiedliche Produktions-/Verbrauchsprofile:

- **Nord & West:** Ganzjährige Defizit-Zonen (ZH, TG, SG / VD, GE) — importieren konstant
- **Mitte:** AKW-Gürtel (AG, BE) — Grundlastüberschuss durch Kernkraft
- **Süd & Ost:** Wasserkraft-Exportzonen (VS, TI, GR) — saisonal stark variierend


In [ ]:
check_aktiv('K_01')  # Kür-Abhängigkeit prüfen



In [ ]:
show('kuer_k01_karte_gegenueberstellung.png',
     'Räumliche Gegenüberstellung: ① Verbrauch · ② Erzeugung · ③ Netzimbalance + Engpässe', width=1100)
show('kuer_k01_karte_erzeuger_verbrauch_detail.png',
     'Erzeuger, Verbrauchszentren & Engpasskorridore — detaillierte Einzelkarte', width=1000)
show('kuer_k01_karte_zonenimbalance.png',
     'Zonenimbalance: Mittlere Produktion (CF-gewichtet) vs. Verbrauch [MW]', width=900)


### 5.2 Battery Value Index (BVI)

Der [BVI](../organisation/O_02_Glossar.ipynb#g-bvi) kombiniert Imbalance-Grösse und Engpassnähe zu einem standortspezifischen Wertindex. Er zeigt wo eine Batterie den grössten kombinierten wirtschaftlichen und systemischen Nutzen hat:

- **Süd** (BVI ~44%): Göschenen-Airolo-Engpass, grosse Wasserkraft-Überschüsse
- **Ost** (BVI ~25%): Alpenwasserkraft Graubünden, Nord-Süd-Exportachse
- **Mitte** (BVI ~20%): AKW-Standorte, Transit-Drehscheibe
- **West** (BVI ~7%): PST-Massnahmen Westgrenze
- **Nord** (BVI ~5%): Verbrauchsmaximum, importabhängig

**Achtung:** Hoher BVI ≠ hohe Arbitrage-Rendite. Süd/Ost haben hohen BVI aber geringeren Spread als Nord/West — der Gesamtwert muss beide Dimensionen einbeziehen.


In [ ]:
show('kuer_k01_bvi_jahresdurchschnitt.png',
     'Battery Value Index (BVI) pro Zone — Jahresdurchschnitt', width=900)


### 5.3 Saisonale BVI-Verschiebung

Der [BVI](../organisation/O_02_Glossar.ipynb#g-bvi) ist nicht statisch — er verschiebt sich mit den Jahreszeiten. Das verändert die optimale Standortempfehlung je nach Betriebsstrategie:


In [ ]:
show('kuer_k01_bvi_heatmap.png',
     'Saisonaler BVI: Optimaler Batteriestandort nach Jahreszeit', width=900)


**Interpretation:**

| Saison | Beste Zone | Grund |
|--------|-----------|-------|
| Winter | Nord / West | Höchstes Netzdefizit; Intraday-Spread aber *niedriger* als Frühling |
| **Frühling** | **Süd / Ost** | **Höchste Marktpreisspanne (~139 EUR/MWh, max−min) — Duck-Curve-Effekt maximal** |
| Sommer | Süd / Ost | Maximaler Überschuss, viel Pufferbedarf |
| Herbst | Mitte | Übergangsphase, AKW-Revision beendet |

**Für Investoren mit fixem Standort** empfehlen wir **Nord (ZH-Raum)**: stabiles Ganzjahres-Netzdefizit = ganzjährig relevanter Systemwert. Hinweis: höchste Marktpreisspanne (max−min) im Frühling (~139 EUR/MWh), nicht im Winter (~85 EUR/MWh) — Nord/West profitiert primär vom konstanten Defizit, nicht vom saisonalen Spread.


---
## 6. Skalierungsszenarien: Vom Einzelprojekt zum Virtual Power Plant <a id='6-skalierungsszenarien-vom-einzelprojekt-zum-virtual-power-plant_K_00'></a>

[↑ Inhaltsverzeichnis](#toc_K_00)

Wenn viele Batterien koordiniert dispatchen, entsteht ein aggregierter Netzentlastungseffekt der über die Einzelanlage hinausgeht. Die vier Szenarien zeigen das Systempotzenzial:


In [ ]:
check_aktiv('K_01')  # Kür-Abhängigkeit prüfen



In [ ]:
show('nb04_spitzenlast.png', 'Systemwirkung: Spitzenlast je Rollout-Szenario', width=850)
show('nb04_spitzenlast_reduktion.png', 'Spitzenlast-Reduktion [%] je Szenario', width=850)


In [ ]:
if _gz_mode != 'unbekannt':
    print(f'Gleichzeitigkeit (aus NB2): {_gz_mode} ({_gz_rate})')
    other = 'optimistisch (70%)' if _gz_mode == 'realistisch' else 'realistisch (40%)'
    factor = 70/40 if _gz_mode == 'realistisch' else 40/70
    print(f'  Zum Vergleich: {other} ergibt {factor:.2f}× dieser Entlastungswerte.')
    print('  Schalter: NB2 → GLEICHZEITIGKEIT = ... → NB2 + NB3 neu ausführen')



**Fazit Skalierung** (Zahlen: NB02 Sektion 5, Gleichzeitigkeit realistisch 40%):

| Szenario | Entlastung | Reduktion | Swissgrid-Relevanz |
|---|---|---|---|
| Status Quo (2024) | 0 MW | 0.0% | — |
| Moderat (2027) | 140 MW | 1.3% | Kaum messbar |
| **Ambitioniert (2030)** | **560 MW** | **5.3%** | Spürbar, aber noch nicht systemkritisch |
| Transformativ (2035) | 2 120 MW | 20.2% | Systemrelevant — koordinierter [Dispatch](../organisation/O_02_Glossar.ipynb#g-dispatch) nötig |

- Erst ab **Transformativ (800k Privat + 2k Industrie, 2035)** wird die Systemwirkung für Swissgrid wirklich relevant (20.2% Reduktion)
- Das Ambitioniert-Szenario (2030) liefert 5.3% — spürbar, aber noch nicht ausreichend für koordinierte Netzplanung
- [BVI](../organisation/O_02_Glossar.ipynb#g-bvi)-gewichtete Verteilung erbringt gegenüber naiver Gleichverteilung +66% Mehrwert — d.h. Standort entscheidet massgeblich über Netzwirkung
- Die Sprünge von Moderat → Ambitioniert → Transformativ zeigen: ab einer kritischen Masse kippt der Systemeffekt in eine neue Grössenordnung


---
## 7. Virtual Power Plant (VPP) — Systemwirkung & Privat-Business-Case <a id='7-virtual-power-plant-vpp-systemwirkung-privat-business-case_K_00'></a>

[↑ Inhaltsverzeichnis](#toc_K_00)

> **Verbindung zu Sektion 6:** Die Skalierungsszenarien zeigen ab wann aggregierte
> Batterien systemrelevant werden. [VPP](../organisation/O_02_Glossar.ipynb#g-vpp) ist der Koordinationsmechanismus der diese
> Skalierung erst ermöglicht — und gleichzeitig der Schlüssel zum Privat-Business-Case.

### 7.1 Das Grundproblem: Privat-Arbitrage allein reicht nicht

Die [Dispatch](../organisation/O_02_Glossar.ipynb#g-dispatch)-Simulation mit realen [ENTSO-E](../organisation/O_02_Glossar.ipynb#g-entsoe)-Preisdaten 2023/2024 zeigt:
Bei einem [CAPEX](../organisation/O_02_Glossar.ipynb#g-capex) von ~4'000 EUR (400 EUR/kWh × 10 kWh) und einem
Netto-Jahreserlös von <100 EUR aus reiner Arbitrage ergibt sich kein
wirtschaftlich tragbarer Business Case für Privathaushalte.

**Das bedeutet nicht, dass Batterien nie rentabel sind** — sondern dass
zwei Faktoren entscheidend sind: der Spread-Level und der CAPEX.

### 7.2 Spread-Entwicklung: Monitoring ist entscheidend


### 7.3 Der FCR-Mechanismus: Bezahlung für Verfügbarkeit, nicht für Energie

Frequenzhaltungsreserve (FCR) funktioniert grundlegend anders als Arbitrage —
und das ist der Grund für den scheinbar überraschenden Rentabilitätssprung zwischen
reiner Arbitrage und [Revenue Stacking](../organisation/O_02_Glossar.ipynb#g-erloess-stacking):

**Arbitrage** zahlt pro verschobener Kilowattstunde. Ohne Laden und Einspeisen kein Erlös.

**FCR** zahlt für das blosse *Vorhalten* von Regelleistung. Swissgrid braucht die
Garantie: Diese Kapazität kann innerhalb von 30 Sekunden plus/minus X kW liefern oder aufnehmen.
Dafür fliesst die Kapazitätsprämie — unabhängig davon ob tatsächlich etwas passiert.

**Grenzfall:** Bleibt die Netzfrequenz zufällig exakt 50.000 Hz, wird nie aktiviert.
Die Batterie steht still — die Prämie fliesst trotzdem.

**In der Praxis** aktiviert [FCR](../organisation/O_02_Glossar.ipynb#g-fcr) zwar häufig, aber näherungsweise symmetrisch:
ungefähr gleich viel Laden wie Einspeisen. Der Netto-Energiefluss über einen Tag
nähert sich gegen null — was für die Batterie sogar vorteilhaft ist:
weniger Vollzyklen → weniger [Degradation](../organisation/O_02_Glossar.ipynb#g-degradation) → längere Lebensdauer.

**Zahlenbeispiel Privat 10 kWh (Mitte-Szenario):**

| Erlösquelle | Mechanismus | Jahreserlös |
|---|---|---|
| Arbitrage (rein) | ~50 Zyklen × 23.87 EUR/MWh × 10 kWh × 92% | ~127 EUR |
| **FCR allein** | **50 EUR/kWh/Jahr × 10 kWh — reine Verfügbarkeitsprämie** | **~500 EUR** |
| [aFRR](../organisation/O_02_Glossar.ipynb#g-afrr) | 30 EUR/kWh/Jahr × 10 kWh | ~300 EUR |
| [VPP](../organisation/O_02_Glossar.ipynb#g-vpp)-Prämie | 25 EUR/kWh/Jahr × 10 kWh | ~250 EUR |
| Smart Tariff | 12 EUR/kWh/Jahr × 10 kWh | ~120 EUR |
| **Total mit Stacking** | | **~1'297 EUR/Jahr** |

FCR allein bringt das **4-fache der gesamten Arbitrage** — ohne einen einzigen Ladevorgang.
Das erklärt den Sprung von Break-Even >50 Jahre (nur Arbitrage) auf ~3–5 Jahre (mit Stacking).

**Warum zahlt der Markt so viel?**
Batterien regeln Frequenzabweichungen in unter 1 Sekunde aus —
Pumpspeicher brauchen ~30 Sekunden, Gaskraftwerke mehrere Minuten.
Diese technische Überlegenheit ist genau das, wofür der Markt 30–80 EUR/kWh/Jahr zahlt.

**Revenue Stacking in der Praxis:** Dieselbe Batterie kann gleichzeitig FCR vorhalten,
in den Lücken zwischen FCR-Aktivierungen Arbitrage betreiben und den Eigenverbrauch
optimieren. Nicht drei Batterien — eine Hardware, drei Erlösströme.

> ⚠️ **CH-Einschränkung:** FCR-Direktteilnahme erfordert eine Mindestgebotskapazität
> von ~1 MW. Ein einzelner Privat-Heimspeicher (5–10 kW) kann **nicht direkt** am
> FCR-Markt teilnehmen — nur aggregiert via VPP-Aggregator.
> Swissgrid Flexibilitätsmarkt für aggregierte Kleinspeicher: **in Aufbau (~2026–2028)**.
> Die Literaturwerte (30–80 EUR/kWh/J) basieren auf DE/UK-Marktpreisen
> (Next Kraftwerke, Octopus Energy) und gelten als Orientierungsgrösse für das
> CH-Potenzial — nicht als heute realisierbare Werte für Privathaushalte.
> Vollständige Stacking-Berechnung → [K_05 Revenue Stacking](K_05_Revenue_Stacking.ipynb).


In [ ]:
check_aktiv('K_03', 'K_05', 'K_99')  # Kür-Abhängigkeit prüfen



**Empirische Belege (K_03/K_05/K_99):** Spread-Zeitreihe, [VPP](../organisation/O_02_Glossar.ipynb#g-vpp)-Erlöse und Kombi-Simulation
validieren den Business Case mit realen Daten — sofern die Kür-Notebooks aktiv sind.


In [ ]:
# Chart 7a: Spread-Zeitreihe als Marktindikator (K_03)
if os.path.exists(os.path.join(CHARTS_DIR, 'kuer_k03_spread_zeitreihe.png')):
    show('kuer_k03_spread_zeitreihe.png',
         'Spread-Entwicklung über Datenzeitraum — monatlicher Intra-Tag-Spread, '
         'Volatilität und Negativpreise. Trendlinie zeigt Richtung des Arbitrage-Potenzials.',
         width=1050)
else:
    print('kuer_k03_spread_zeitreihe.png nicht vorhanden → K_03 ausführen.')

# Chart 7b: FCR als Gamechanger — Break-Even und ROI pro Segment (K_05)
if os.path.exists(os.path.join(CHARTS_DIR, 'kuer_k05_fcr_gamechanger.png')):
    show('kuer_k05_fcr_gamechanger.png',
         'FCR als Gamechanger: Break-Even und ROI pro Segment — '
         'Nur Arbitrage vs. +FCR vs. +FCR+aFRR vs. Voll-Stacking (K_05 Mitte-Szenario)',
         width=1050)
else:
    print('kuer_k05_fcr_gamechanger.png nicht vorhanden → K_05 ausführen.')

# Chart 7c: Erlösstacking und CAPEX-Szenarien (K_99)
show('kuer_k99_revenue_split.png',
     'Erlösstacking: Wie setzt sich der Gesamterlös zusammen?', width=1000)
show('kuer_k99_capex_szenarien.png',
     'CAPEX-Szenarien: ROI und Break-Even bei Heute / Trigger / Ziel', width=1050)



### 8.3 CAPEX-Lernkurve trifft steigenden Spread: Das Fenster öffnet sich

Zwei Trends wirken gleichzeitig in die richtige Richtung:

| Faktor | Historische Rate | Projektion |
|---|---|---|
| **CAPEX Li-Ion** | −10–15%/Jahr (2015–2024) | 2027: ~250 EUR/kWh, 2030: ~180 EUR/kWh |
| **Intra-Tag-Spread CH** | steigt mit EE-Ausbau | +5–15% jährlich (Schätzung) |

Die folgende Heatmap zeigt Break-Even-Jahre für alle [CAPEX](../organisation/O_02_Glossar.ipynb#g-capex) × Spread Kombinationen:


In [ ]:
# Chart 8: Sensitivitätsheatmap (K_03)
if os.path.exists(os.path.join(CHARTS_DIR, 'kuer_k03_sensitivitaet_heatmap.png')):
    show('kuer_k03_sensitivitaet_heatmap.png',
         'Break-Even-Jahre als Funktion von CAPEX und Intra-Tag-Spread (Privat 10kWh/5kW). '
         'Horizontale Linien: CAPEX-Lernkurve. Vertikale Linie: aktueller CH-Spread.',
         width=1050)
else:
    print('kuer_k03_sensitivitaet_heatmap.png nicht vorhanden → K_03 ausführen.')

# → Kombinationsmatrix vollständig in Abschnitt 10



### 8.4 VPP als Erlös-Multiplikator

Auch bei aktuellem Spread und [CAPEX](../organisation/O_02_Glossar.ipynb#g-capex) verbessert [Revenue Stacking](../organisation/O_02_Glossar.ipynb#g-erloess-stacking) den Business Case fundamental.
Der entscheidende Unterschied: [FCR](../organisation/O_02_Glossar.ipynb#g-fcr) zahlt für **Verfügbarkeit**, nicht für gelieferte Energie
(→ Sektion 7.3). Die Batterie kann gleichzeitig Arbitrage betreiben und FCR vorhalten.

| Erlösquelle | Mechanismus | Zusatz [EUR/kWh/J] | Heute CH |
|---|---|---|---|
| **Eigenverbrauch** | EV-/WP-Optimierung | +~10 | ✅ Heute möglich |
| **Smart Tariff** | Netzentgeltsenkung bei Lastkoordination | +12 | ✅ Pilotphase |
| **FCR** | Frequenzregelung — Verfügbarkeitsprämie | +30–80 | ⏳ via [VPP](../organisation/O_02_Glossar.ipynb#g-vpp) ~2026–28 |
| **aFRR** | Koordinierter [Dispatch](../organisation/O_02_Glossar.ipynb#g-dispatch) auf Swissgrid-Signal | +20–50 | ⏳ via VPP ~2026–28 |
| **VPP-Kapazitätsprämie** | Aggregierte Leistungsbereitstellung | +20–40 | ⏳ in Aufbau |

**Wirkung auf Break-Even Privat (10 kWh, 350 EUR/kWh, 20 EUR/MWh):**

| Strategie | Break-Even | Status |
|---|---|---|
| Nur Arbitrage | >100J | ❌ |
| + Eigenverbrauch | ~27J | ⚠️ heute machbar |
| + EV + Smart Tariff | ~14J | ⚠️ knapp ausserhalb 12J-Grenze |
| + FCR via VPP | ~7J | ✅ sobald Swissgrid öffnet |

> Quantitative Herleitung → [K_05 Revenue Stacking](K_05_Revenue_Stacking.ipynb), Chart FCR-Gamechanger.  
> Vier-Panel-Heatmap nach Strategie → Sektion 10.1 unten.

### 8.5 Internationale Referenzmodelle

| Land | Modell | Status |
|---|---|---|
| **UK** | Octopus Powerloop, OVO Drive & Charge | Operativ |
| **DE** | Next Kraftwerke, Sonnen Community | Operativ |
| **AT** | Wien Energie Flex Pool | Pilotphase |
| **CH** | Swissgrid Flexibilitätsmarkt | In Aufbau (~2026–2028) |

### 8.6 Empfehlung: Jetzt beobachten, beim Trigger investieren

> **Der richtige Zeitpunkt für Privat-Investitionen:** Wenn CAPEX < 250 EUR/kWh
> **und** CH-Monatsmedian-Spread > 30 EUR/MWh **und** Swissgrid Flexibilitätsmarkt
> für aggregierte Heimspeicher geöffnet. Chart 7a (jährlich neu generieren) zeigt
> ob der Spread-Trigger erreicht ist. Chart 8 zeigt den Break-Even für den
> dann gültigen CAPEX.


---
## 8. Empirische Validierung <a id='8-empirische-validierung_K_00'></a>

[↑ Inhaltsverzeichnis](#toc_K_00)

Der Business Case ist nicht nur theoretisch — er wird durch reale Grenzflussdaten bestätigt. CH importiert Strom genau in den Stunden mit dem höchsten Preis, und die Batterie-[Dispatch](../organisation/O_02_Glossar.ipynb#g-dispatch)-Logik greift in einem Grossteil dieser Import-Stunden. [Grid-Arbitrage](../organisation/O_02_Glossar.ipynb#g-grid-arbitrage) und [Netzentlastung](../organisation/O_02_Glossar.ipynb#g-netzentlastung) wirken damit **gleichzeitig** in die gleiche Richtung.


In [ ]:
check_aktiv('K_02')  # Kür-Abhängigkeit prüfen



**Import/Export-Validierung (K_02):** Schweizer Grenzflüsse zeigen, dass CH in
Hochpreis-Stunden netto importiert — direkte empirische Bestätigung des Arbitrage-Musters.


In [ ]:
# ── Sektion 7: Import/Export-Validierung (Chart 6) ───────────────────────────
# Chart 6 wird in K_02 Cross-Border (Kür) erzeugt → output/charts/<SZ_AKTIV>/
if os.path.exists(os.path.join(CHARTS_DIR, 'kuer_k02_import_export.png')):
    show('kuer_k02_import_export.png',
         'Import/Export-Validierung: CH kauft Strom genau wenn Batterien einspeisen sollten',
         width=900)
else:
    print('ℹ️  Chart Import/Export nicht vorhanden → K_02 zuerst ausführen.')



---


---
## 9. Handlungsempfehlungen nach Stakeholder <a id='9-handlungsempfehlungen-nach-stakeholder_K_00'></a>

[↑ Inhaltsverzeichnis](#toc_K_00)

### Für Investoren und Projektierer

1. **Standort zuerst:** Utility-Projekt in Süd/Ost für saisonale Arbitrage im Sommer,    in Nord für ganzjährig stabile Rendite
2. **Erlösstacking:** [Grid-Arbitrage](../organisation/O_02_Glossar.ipynb#g-grid-arbitrage) als Basiserlös, FCR/aFRR für Industrie/Utility    als primärer [ROI](../organisation/O_02_Glossar.ipynb#g-roi)-Treiber
3. **Saisonal optimierter [Dispatch](../organisation/O_02_Glossar.ipynb#g-dispatch):** Ladefenster im Sommer auf Mittag verschieben,    Winter-Dispatch auf frühe Morgenstunden konzentrieren → +10–20% ROI
4. **Aggregation anstreben:** Ab 10+ Anlagen ist ein koordinierter Dispatch    (Virtual Power Plant) wirtschaftlich überzeugend

### Für Netzbetreiber und Swissgrid

1. **BVI-basierte Förderprogramme:** Batterien an Engpässen (Göschenen-Airolo,    AG-ZH Mittelland) mit Netzentgeltsenkungen incentivieren
2. **Saisonale Ausgleichsmechanismen:** Sommerpuffer in Süd/Ost explizit vergüten
3. **Digitale Plattform:** [API](../organisation/O_02_Glossar.ipynb#g-api) für koordinierten Dispatch-Abruf statt Einzelverträge

### Für Politik und BFE

1. **Förderung dezentraler Speicher** über Netzentgeltsenkungen für nachweisbares Peak-Shaving
2. **Regulatorischer Rahmen** für Privat-Batterien als Systemdienstleister
3. **Datenzugang:** [ENTSO-E](../organisation/O_02_Glossar.ipynb#g-entsoe) Echtzeit-API und swissgrid-Engpassdaten für Marktakteure öffnen


---
## 10. Kombinationsmatrix: Parameter-Szenarien <a id='10-kombinationsmatrix-parameter-szenarien_K_00'></a>

[↑ Inhaltsverzeichnis](#toc_K_00)

Welche Parameterkombination ergibt welchen Business Case?

| Spread | [CAPEX](../organisation/O_02_Glossar.ipynb#g-capex) | [Dispatch](../organisation/O_02_Glossar.ipynb#g-dispatch) | Standort-[BVI](../organisation/O_02_Glossar.ipynb#g-bvi) | Break-Even Privat | Empfehlung |
|--------|-------|----------|--------------|-------------------|------------|
| Heute (~20 EUR/MWh) | Heute (~350 EUR/kWh) | Reaktiv | Beliebig | > 25 J | ❌ Nicht investieren |
| Heute (~20 EUR/MWh) | Heute (~350 EUR/kWh) | DA-optimal | Hoch (Süd/Nord) | ~22 J | ❌ Abwarten |
| Heute (~20 EUR/MWh) | Trigger (250 EUR/kWh) | DA-optimal | Hoch | ~15 J | ⚠️ Grenzwertig |
| Trigger (30 EUR/MWh) | Heute (~350 EUR/kWh) | Reaktiv | Beliebig | ~20 J | ⚠️ Grenzwertig |
| Trigger (30 EUR/MWh) | Trigger (250 EUR/kWh) | DA-optimal | Mittel | ~10 J | ✅ Investieren |
| Trigger (30 EUR/MWh) | Trigger (250 EUR/kWh) | [VPP](../organisation/O_02_Glossar.ipynb#g-vpp)-Stacking | Hoch | ~6–8 J | ✅✅ Ideal |
| Hoch (50 EUR/MWh) | Ziel (180 EUR/kWh) | VPP-Stacking | Hoch | ~4–5 J | ✅✅✅ Skalieren |

> **Lesehilfe:** Break-Even Privat (10 kWh / 400 EUR/kWh CAPEX heute). Spread = Intra-Tag-Spread (p75−p25), Basis der Simulation — nicht die rohe Marktpreisspanne (max−min, bis ~139 EUR/MWh). Werte skalierend —
> Industrie/Utility ist in jeder Zeile ca. 2–3× besser als Privat.

**Wann treten die Trigger ein?**
- Spread > 30 EUR/MWh: bei weiterem EE-Ausbau CH (~2026–2028 laut K_03 Trend)
- CAPEX < 250 EUR/kWh: laut NREL ATB Advanced-Szenario ~2025–2026 (K_07)
- VPP-Stacking: Swissgrid Flexibilitätsmarkt in Aufbau (~2026–2028)

> Vollständige Parametersensitivität → K_03 Chart 08-B (CAPEX × Spread Heatmap)  
> Technologieabhängigkeit → K_07 Eignungsmatrix  
> [Erlösstacking](../organisation/O_02_Glossar.ipynb#g-erloess-stacking)-Details → K_05


### 10.1 Strategievergleich: Vier Wege zur Rentabilität

Die Heatmap-Panels unten zeigen vier differenzierte Strategien — von der reinen
Arbitrage-Baseline bis zum [FCR](../organisation/O_02_Glossar.ipynb#g-fcr)-Kipppunkt. Jedes Panel verwendet dieselbe Achsenskala
(Intraday-Spread × System-CAPEX), die weisse gestrichelte Linie markiert die 12J-Grenze.

| Panel | Strategie | Zusatz [EUR/kWh/J] | Heute-BE | Felder <12J | Heute machbar? |
|---|---|---|---|---|---|
| P1 | Nur Arbitrage | 0 | >100J | 6/49 | ✅ (aber nicht rentabel) |
| P2 | + Eigenverbrauch | ~10.5 | ~27J | 25/49 | ✅ keine Voraussetzungen |
| P3 | + EV + Smart Tariff | ~22.5 | ~14J | 40/49 | ✅ regulatorisch heute möglich |
| P4 | + FCR via [VPP](../organisation/O_02_Glossar.ipynb#g-vpp) | ~50 | ~7J | 49/49 | ⏳ Swissgrid ~2026–28 |

**P3 ist der interessanteste Befund:** Ohne FCR, ohne Regelenergiemarkt — nur mit
Eigenverbrauchsoptimierung und Netzentgeltsenkung — werden bereits 40 von 49
Parameterkombinationen rentabel. Beim Trigger-Punkt (30 EUR/MWh, 250 EUR/kWh)
ergibt sich Break-Even **8.5J** — innerhalb der 12J-Grenze.

**P4 (FCR)** dreht dann das letzte Drittel — und zwar schlagartig: 6 → 49 Felder
in einem Schritt. Das zeigt warum Swissgrid-Marktöffnung ein Wendepunkt ist.

> FCR-Mechanismus (Verfügbarkeitsprämie, kein Energieerlös) → Sektion 7.3  
> Quantifizierung pro Segment → [K_05 Revenue Stacking](K_05_Revenue_Stacking.ipynb) Chart FCR-Gamechanger


In [ ]:
# ── Kombinationsmatrix: 2x2 Übersicht + 4 Einzelpanels (K_99) ────────────────
show('kuer_k99_kombi_heatmap.png',
     'Vier Strategien im Vergleich: Nur Arbitrage → EV → EV+SmartTariff → FCR (K_99)',
     width=1100)

panel_info = [
    ('kuer_k99_kombi_heatmap_p1.png',
     'P1 — Nur Arbitrage (Baseline): 6/49 Felder rentabel — fast alles rot'),
    ('kuer_k99_kombi_heatmap_p2.png',
     'P2 — + Eigenverbrauch (~10.5 EUR/kWh/J): 25/49 Felder — heute ohne Voraussetzungen realisierbar'),
    ('kuer_k99_kombi_heatmap_p3.png',
     'P3 — + EV + Smart Tariff (~22.5 EUR/kWh/J): 40/49 Felder — '
     'heute regulatorisch möglich, Trigger-Punkt bei 8.5J'),
    ('kuer_k99_kombi_heatmap_p4.png',
     'P4 — + FCR via VPP (~50 EUR/kWh/J): 49/49 Felder — Kipppunkt, Swissgrid ~2026-28'),
]
for fname, caption in panel_info:
    if os.path.exists(os.path.join(CHARTS_DIR, fname)):
        show(fname, caption, width=1050)
    else:
        print(f'⚠️  {fname} fehlt → K_99 neu ausführen')



---
## 11. Strategisches Fazit <a id='11-strategisches-fazit_K_00'></a>

[↑ Inhaltsverzeichnis](#toc_K_00)

[Grid-Arbitrage](../organisation/O_02_Glossar.ipynb#g-grid-arbitrage) mit Batteriespeichern ist in der Schweiz **wirtschaftlich machbar und systemisch wertvoll** — aber der Gesamtnutzen hängt stark von drei Variablen ab:

**Wer:** Industrie und Utility profitieren klar von reiner Arbitrage. Privat und Gewerbe brauchen [Erlösstacking](../organisation/O_02_Glossar.ipynb#g-erloess-stacking) (Eigenverbrauch, Peak-Shaving, FCR/aFRR).

**Wann:** Frühling liefert die höchste Marktpreisspanne (~139 EUR/MWh, max−min) — entgegen intuitiver Erwartung. Der Duck-Curve-Effekt ist im Frühling am stärksten (Solar bereits aktiv, Heizlast noch hoch → grosser Tagesgangunterschied). Winter hat den niedrigsten Spread (~85 EUR/MWh). Sommer-Mittag bietet Negativpreis-Ladezyklen. Ein adaptiver Algorithmus verbessert den Jahres-[ROI](../organisation/O_02_Glossar.ipynb#g-roi) um 10–20% gegenüber statischem [Dispatch](../organisation/O_02_Glossar.ipynb#g-dispatch).

**Wo:** Für maximale Systemwirkung: Engpasszonen Süd/Ost. Für robuste Ganzjahresrendite: Nord (ZH-Raum) als Defizit-Zone mit konstantem Netzdefizit — ganzjährig stabile Systemrelevanz. [BVI](../organisation/O_02_Glossar.ipynb#g-bvi)-gewichteter Rollout bringt gegenüber naiver Verteilung +66% Netzentlastungseffekt.

---
*Erstellt im Rahmen des CAS Information Engineering – Scripting, ZHAW School of Engineering.*

> **Weiterführende Kür-Notebooks:**  
> [K_07 Technologievergleich](K_07_Technologievergleich.ipynb) · [K_08 Alternative Speicher](K_08_Alternative_Speicher.ipynb) · [K_09 Eigenverbrauchsopt.](K_09_Eigenverbrauch.ipynb) · [K_10 Produktsteckbrief](K_10_Produkt_Steckbrief.ipynb)  
*Daten: [ENTSO-E](../organisation/O_02_Glossar.ipynb#g-entsoe) (DL-DE-BY-2.0) · [BFE](../organisation/O_02_Glossar.ipynb#g-bfe) (Open Government Data) · [swisstopo](../organisation/O_02_Glossar.ipynb#g-swisstopo) swissBOUNDARIES3D · [BFS](../organisation/O_02_Glossar.ipynb#g-bfs) [STATPOP](../organisation/O_02_Glossar.ipynb#g-statpop).*


---
## Abschluss <a id='abschluss_K_00'></a>

[↑ Inhaltsverzeichnis](#toc_K_00)

Szenario und Verfügbarkeit aller referenzierten Charts bestätigen.


In [ ]:
# ── Abschlusskontrolle K_00 ──────────────────────────────────────────────────
print('K_00 – Abschlusskontrolle')
print('=' * 60)
print('K_00 ist ein reines Report-Notebook — erzeugt keine eigenen Chart-Dateien.')
print('Alle angezeigten Charts stammen aus NB03/K_01/K_02/K_03/K_05/K_99.')
print(f'Szenario: {SZ_AKTIV}')
print('→ Weiter mit Kür-Notebooks falls aktiv, sonst direkt NB99.')



| [← K_99 – Kombinierte Simulation](K_99_Kombinierte_Simulation.ipynb) | [↑ Übersicht](../organisation/O_01_Project_Overview.ipynb) | [K_01 – Räumliche Analyse →](K_01_Raeumliche_Analyse.ipynb) |
|:---|:---:|---:|